## <font color="#FF9900"> Fluxo do Kafka local (Docker)</font>
<font size="3" color="gray">

optei por utilizar o Docker para rodar o Kafka no meu próprio computador antes de subir para o AWS
utilizei o docker-compose.yml e o serviço de WSL para criar um servidor local e rodar o kafka na minha própria máquina. <br>
Vou deixar aqui um tutorial de instalação do WSL e o link para o Docker desktop caso alguem queira tentar usar. <br>

mas na prática utilizando o confluent-kafka ao modificar o ip do localhost para o novo ip da aws nós temos acesso ao kafka na nuvem sem grande dificuldade de configuração. 

ponto positivo pra engenharia moderna ! 


</font>



[AULA WSL](https://www.youtube.com/watch?v=trto4i0Olwg&t=2s)  <BR>
[DOCKER DESKTOP](https://www.docker.com/products/docker-desktop/)


In [ ]:

#Instalar o Confluent kafka para utilização da ferramenta e fazer os imports necessarios 
%pip install confluent-kafka
%pip install  pyspark

import os
import time
import json
import random
import boto3
import dotenv
from datetime import datetime
import pandas as pd
from pathlib import Path

dotenv.load_dotenv()

from confluent_kafka.admin import AdminClient, NewTopic
from confluent_kafka import Producer,Consumer, KafkaError

from src.data.utils import enriquecer_alunos_silver, carregar_parquet_local, preparar_dimensoes_silver,iniciar_cessao_aws

In [ ]:
#podemos rodar novamente para limpar as mensagens 

server = os.getenv("SERVER_KAFKA", "localhost:9092") #Mantemos o localhost caso nao de certo o SERVER_KAFKA
admin_client = AdminClient({'bootstrap.servers': server})

# Apaga o tópico com todas as mensagens acumuladas
fs = admin_client.delete_topics(['transactions'], operation_timeout=15)

for topic, future in fs.items():
    try:
        future.result()
        print(f"'{topic}' deletado!")
    except Exception as e:
        print(f"{e}")

# Aguarda o Kafka processar a deleção internamente

time.sleep(3)

#  Recria o tópico limpo com 3 partições
print("Recriando tópico 'transactions'")
fs = admin_client.create_topics([NewTopic("transactions", num_partitions=3, replication_factor=1)])
for topic, future in fs.items():
    try:
        future.result()
        print(f"Tópico '{topic}' recriado")
    except Exception as e:
        print(f"{e}")


In [ ]:
# Configurações do Kafka Local
TOPIC_NAME = "transactions"
PATH_BRONZE = Path("../data/bronze")
PATH_PRATA = Path("../data/prata")
#ANO = 2023 substitui para rodar um for 
KAFKA_SERVER = os.getenv("SERVER_KAFKA", "localhost:9092")
TOPIC_NAME = "transactions"
PATH_BRONZE = Path("../data/bronze")



In [ ]:
def delivery_report(err, msg):
    """
    Retorna um log de envio ou erro do producer Kafka 
    """

    if err is not None:
        print(f"[ERRO] Falha no envio: {err}")
    # else:
    #     print(f"[SUCESSO] Aluno enviado para {msg.topic()} | Partição: {msg.partition()} | ID Aluno: {msg.key().decode('utf-8')}")


<font  size="3" color="gray">

Irei configurar o Producer 
que é responsavel por enviar a mensagem para o Kafka <br>
Para isso vamos simular um serviço em streaming, em que a cada segundo um aluno entrega a prova, 
<br> criando uma linha no TS_ALUNO 

O Objetivo dessa parte é carregar o dataframe de alunos e enviar para o Kafka 
O Script faz o papel de producer nesse caso 
ao chegar o kafka o consumer recuperará as linhas geradas pelo producer 
irá ler cada uma delas e fazer o pivot table com o municipio e o aluno, gerando assim a camada silver 

</font>

In [ ]:
# Definir o ano de processamento (2023, 2024 ou 2025)
ANOS_PROVAS = [2024]  # 2023,2024,2025

# Carrega o arquivo inteiro de alunos do ano escolhido
for ANO in ANOS_PROVAS:
    caminho_parquet = PATH_BRONZE / f"ano={ANO}/dados/TS_ALUNO.parquet"
    print(f"[INFO] Carregando base de alunos do ano {ANO} de: {caminho_parquet.name}...")
    df_alunos_source = pd.read_parquet(caminho_parquet)
    #df_alunos_source = df_alunos_source.head(20000)  # utilizado para limitar para testes mais rápidos

    total_registros = len(df_alunos_source)
    print(f"Total de registros para enviar ao Kafka: {total_registros}")

    producer = Producer({'bootstrap.servers': KAFKA_SERVER})

    print(f"\n Iniciando streaming do ano {ANO}...")

    try:
        for index, row in df_alunos_source.iterrows():
            dados_aluno = row.to_dict()
            dados_aluno_limpo = {}
            for k, v in dados_aluno.items():
                if pd.isna(v):
                    dados_aluno_limpo[k] = None
                elif hasattr(v, 'item'):
                    dados_aluno_limpo[k] = v.item()
                else:
                    dados_aluno_limpo[k] = v

            id_aluno = str(dados_aluno_limpo.get('ID_ALUNO') or index)
            
            # Tratamento de Buffer: Evita o erro de Queue Full quando o loop é mais rápido que a rede
            while True:
                try:
                    producer.produce(
                        TOPIC_NAME,
                        key=id_aluno.encode('utf-8'),
                        value=json.dumps(dados_aluno_limpo).encode('utf-8'),
                        callback=delivery_report
                    )
                    break  # Mensagem aceita na fila local, sai do loop de reenvio e vai para o próximo aluno
                except BufferError:
                    # Fila cheia, aguarda 0.5 segundo para a internet enviar as mensagens e liberar espaço
                    producer.poll(0.5)
            
            # Faz o poll a cada 1.000 mensagens para liberar memória interna e disparar callbacks
            if index % 1000 == 0:
                producer.poll(0)
            
            # Imprime progresso consolidado de 10.000 em 10.000 mensagens
            if index % 10000 == 0:
                print(f"[Status] {index}/{total_registros} mensagens enviadas...")

    except KeyboardInterrupt:
        print("\n Envio pausado pelo usuário.")
    finally:
        print(" Finalizando envios pendentes (flush)... Limpando a fila de mensagens...")
        producer.flush()
        print(f" Ingestão concluída com sucesso para o ano {ANO}!")


<font size="3" color="gray">
evidencia das mensagens que foram enviadas para o kafka via ui kafka 
ano 2023 processado via streaming 

Evidencia do envio dos 1747439 de alunos pelo notebook 

![Bucket Criado no S3](../reports/figures/processado_2023_notebook.png)

Kafka populado com os 1747439

![Bucket Criado no S3](../reports/figures/kafka_2023_nuvem.png)

para agilizar o processo os demais anos irei processar como batch, direto via notebook python e ignorando o kafka
(levamos 17 minutos para processar 17 milhões de linhas)

</font>

In [ ]:
def processar_camada_silver(ano: int | str, df_alunos: pd.DataFrame = None, path_bronze: Path = Path("../data/bronze")) -> pd.DataFrame:
    """
    Processa e integra a camada Silver. 
    Se df_alunos for informado, processa-o (Streaming).
    Caso contrário, carrega o arquivo Parquet do disco local (Batch).
    """
    # Se for Batch (df_alunos é None), carrega do disco local
    if df_alunos is None:
        print(f"\n[INFO] Modo Batch: Carregando alunos do ano {ano} do disco...")
        df_alunos = pd.read_parquet(path_bronze / f"ano={ano}/dados/TS_ALUNO.parquet")
        
    # Prepara as tabelas de referência
    municipio_dim, uf_dim = preparar_dimensoes_silver(ano, path_bronze)
    
    # Executa o enriquecimento comum (Junções e Cálculos)
    df_silver = enriquecer_alunos_silver(df_alunos, municipio_dim, uf_dim)
    
    return df_silver


<font size='3' color='gray'>O Consumer opera através das mensagens recebidas pelo Kafka 
</font>


In [ ]:
# Configurações do Micro-batch
BATCH_SIZE_LIMIT = 10000  # LIMITE DE MENSAGENS POR BUFFER
BATCH_TIME_LIMIT = 10     # TEMPO LIMITE PARA FECHAR O BUFFER (segundos)

# Carrega e prepara as dimensões usando o seu utils.py
print("[INFO] Preparando tabelas de referência (Município/UF)...")
municipio_dim, uf_dim = preparar_dimensoes_silver(ANO, PATH_BRONZE)

# Inicializa o cliente do S3 usando a sessão ativa do boto3
session = iniciar_cessao_aws()
s3_client = session.client('s3')
nome_do_bucket = os.getenv("BUCKET_NAME")

# Configura e assina o Consumidor Kafka
consumer = Consumer({
    'bootstrap.servers': KAFKA_SERVER,
    'group.id': 'pipeline-silver',   #esse id é utilizado para salvar quais foram as mensagens lidas até o momento, trocar para reiniciar o processamento do zero
    'auto.offset.reset': 'earliest'  #estrategia do consumidor, dessa forma começa processando as mensagens mais antigas
})
consumer.subscribe([TOPIC_NAME])
print("Consumidor iniciado e escutando o Kafka...")

buffer_mensagens = []
ultimo_envio_tempo = time.time()

# Loop de processamento
total_processado_acumulado = 0  
try:
    while True:
        msg = consumer.poll(0.5)
        
        if msg is not None:
            if msg.error():
                if msg.error().code() != KafkaError._PARTITION_EOF:
                    print(f"[Erro] Kafka: {msg.error()}")
                    break
            else:
                dados_aluno_json = json.loads(msg.value().decode('utf-8'))
                
                colunas_obrigatorias = ['NU_ANO_AVALIACAO', 'CO_UF', 'CO_MUNICIPIO', 'TP_SERIE']
                if all(col in dados_aluno_json for col in colunas_obrigatorias):
                    buffer_mensagens.append(dados_aluno_json)
        
        tempo_decorrido = time.time() - ultimo_envio_tempo
        tamanho_lote = len(buffer_mensagens)
        
        # Quando atingir o limite de tempo ou de quantidade do buffer o lote é processado
        # isso é utilizado para implementar o padrão micro-batching, utilizado para resolver o problema de arquivos pequenos (com poucos alunos)
        # dessa forma nao iremos processar milhares de arquivos pequenos na hora de consumir esses dados, isso geraria custo
        # também é utilizado para controlar a memória, é uma técnica para não processar nem arquivos demais nem de menos 
        if tamanho_lote > 0 and (tamanho_lote >= BATCH_SIZE_LIMIT or tempo_decorrido >= BATCH_TIME_LIMIT):
            df_lote_alunos = pd.DataFrame(buffer_mensagens)
            df_lote_alunos = df_lote_alunos.dropna(subset=colunas_obrigatorias)
            
            if not df_lote_alunos.empty:
                for col_name in colunas_obrigatorias:
                    df_lote_alunos[col_name] = pd.to_numeric(df_lote_alunos[col_name]).astype(int)
                if 'VL_PROFICIENCIA_LP' in df_lote_alunos.columns:
                    df_lote_alunos['VL_PROFICIENCIA_LP'] = pd.to_numeric(df_lote_alunos['VL_PROFICIENCIA_LP'])
                
                # Transformar as tabelas para o padrão Silver (Stream-Static Join)
                df_silver = enriquecer_alunos_silver(df_lote_alunos, municipio_dim, uf_dim)
                
                timestamp_atual = int(time.time())
                nome_arquivo = f"lote_alunos_{timestamp_atual}.parquet"
                
                # 1. Gravação Local (Backup)
                pasta_destino = PATH_PRATA / f"ano={ANO}"
                pasta_destino.mkdir(parents=True, exist_ok=True)
                caminho_arquivo_local = pasta_destino / nome_arquivo
                df_silver.to_parquet(caminho_arquivo_local, index=False)
                
                # 2. Gravação na Nuvem (S3)
                chave_s3 = f"prata/ano={ANO}/{nome_arquivo}"
                try:
                    # Converte o DataFrame para bytes de Parquet na memória
                    import io
                    parquet_buffer = io.BytesIO()
                    df_silver.to_parquet(parquet_buffer, index=False)
                    parquet_bytes = parquet_buffer.getvalue()
                    
                    # Envia os bytes para o S3
                    s3_client.put_object(
                        Bucket=nome_do_bucket,
                        Key=chave_s3,
                        Body=parquet_bytes
                    )
                    total_processado_acumulado += len(df_silver)

                    print(f"[Sucesso] Lote de  {len(df_silver)}  arquivos salvos na Silver local e S3: total de registros processados: {total_processado_acumulado} )")
                except Exception as e:
                    print(f"[Erro] ao enviar lote para o S3 (gravado apenas localmente): {e}")
            else:
                print("O lote continha apenas registros inválidos (com chaves nulas) e foi ignorado.")
            
            # Limpa o buffer para o próximo ciclo
            buffer_mensagens = []
            ultimo_envio_tempo = time.time()

except KeyboardInterrupt:
    print("\n Consumidor parado pelo usuário.")
finally:
    consumer.close()



<font size="3" color="gray">
2023 eu rodei o consumer no próprio notebook <br>
e ocorreu tudo certo conseguimos operar 100% dos dados 

![Bucket Criado no S3](../reports/figures/consumer_2023.png)



<font size="3" color="gray">Rodar no terminal para ler o status das mensagens processadas 
</font> 
<br>
<font size="3" color="green">
docker exec -it kafka-fiap /opt/kafka/bin/kafka-consumer-groups.sh --bootstrap-server localhost:9092 --describe --group pipeline-silver-group-multiyear
</font>

In [ ]:
!docker exec -it kafka-fiap /opt/kafka/bin/kafka-consumer-groups.sh --bootstrap-server localhost:9092 --describe --group pipeline-silver-group-v3


fiz producer.py para subir no EC2, mas vou rodar ele local

In [ ]:
# !python -m src.streaming.producer 2024
# !python -m src.streaming.producer 2025

<font size='4' color="gray"> Evidencias producer 2024

![Bucket Criado no S3](../reports/figures/PRODUCER_2024.png)

Evidencias producer 2025

![Bucket Criado no S3](../reports/figures/PRODUCER_2025.png)

![Bucket Criado no S3](../reports/figures/kafka_producer.py.png)



![Bucket Criado no S3](../reports/figures/LOG_LAMBDA_CONSUMER.png)


![Bucket Criado no S3](../reports/figures/MONITORAMENTO_LAMBDA.png)

![Bucket Criado no S3](../reports/figures/ERROS_CLOUDWATCH.png)

![Bucket Criado no S3](../reports/figures/S3_PRATA.png)

<font size='4' color='gray'>Como o Kafka particiona os arquivos, optei por enviar a minha camada prata local que está apenas em 1 parquet por ano, arquivo que gerei anteriormente no notebook 02 - pipeline_bronze_silver_local. <br>
como a próxima camada pretendo operar na nuvem, diminuir a quantidade de parquet ajudaria a diminuir o custo de get do s3, alem de velocidade operacional. 

In [ ]:
import os
import boto3
from pathlib import Path
from src.data.utils import iniciar_cessao_aws


session = iniciar_cessao_aws()
s3 = session.client('s3')
BUCKET_NAME = os.getenv("BUCKET_NAME")
anos = [2023, 2024, 2025]
 
caminho_base = Path("..")  

for ano in anos:
    caminho_local = caminho_base / "data" / "prata" / f"ano={ano}" / "alunos_prata.parquet"
    chave_s3 = f"prata/ano={ano}/alunos_prata.parquet"
    
    if caminho_local.exists():
        print(f"Fazendo upload de {caminho_local.name} (Ano {ano}) para o S3...")
        try:
            s3.upload_file(
                Filename=str(caminho_local),
                Bucket=BUCKET_NAME,
                Key=chave_s3
            )
            print(f"  [SUCESSO] Enviado para s3://{BUCKET_NAME}/{chave_s3}")
        except Exception as e:
            print(f"  [ERRO] Falha no upload para o ano {ano}: {e}")
    else:
        print(f"  [AVISO] Arquivo não encontrado localmente em: {caminho_local}")


Fazendo upload de alunos_prata.parquet (Ano 2023) para o S3...
  [SUCESSO] Enviado para s3://fiap-postech-challenge-datascience-002/prata/ano=2023/alunos_prata.parquet
Fazendo upload de alunos_prata.parquet (Ano 2024) para o S3...
  [SUCESSO] Enviado para s3://fiap-postech-challenge-datascience-002/prata/ano=2024/alunos_prata.parquet
Fazendo upload de alunos_prata.parquet (Ano 2025) para o S3...
  [SUCESSO] Enviado para s3://fiap-postech-challenge-datascience-002/prata/ano=2025/alunos_prata.parquet


<font size="3" color="gray">

evidencia da quantidade de linhas no parquet silver (print do notebook Silver -> Gold)

![Bucket Criado no S3](../reports/figures/TOTAL_ALUNOS_SILVER.png)

Evidencia da quantidaded e alunos nos csv (Print do notebook 00 - EDA )

![Bucket Criado no S3](../reports/figures/ALUNOS_CSV.png)